In [ ]:
# Test 1: Check Python environment
import sys
print(f"Python version: {sys.version}")
print(f"Python path: {sys.executable}")

In [ ]:
# Test 2: Check CUDA
import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")


In [ ]:

# Test 3: Check key imports
import numpy as np
import pandas as pd
import transformers
import sentence_transformers
import llama_index
import langchain

print(f"\nNumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"Sentence Transformers: {sentence_transformers.__version__}")
print(f"LangChain: {langchain.__version__}")

In [ ]:
import httpx
import json
import numpy as np
import pandas as pd
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from sentence_transformers import SentenceTransformer

# Test all services
print("Testing all services...")
services = {
    "LLM Server": "http://localhost:8001/health",
    "Embedding Server": "http://localhost:8002/health",
    "Qdrant": "http://localhost:6333"
}

for name, url in services.items():
    try:
        if name == "Qdrant":
            resp = httpx.get(url, timeout=5)
            status = "✓" if resp.status_code == 200 else "✗"
        else:
            resp = httpx.get(url, timeout=5)
            data = resp.json()
            status = "✓" if data.get("status") == "healthy" else "✗"
        print(f"{name}: {status}")
    except Exception as e:
        print(f"{name}: ✗ ({str(e)[:50]})")


In [ ]:
from datasets import load_dataset

eval_data = load_dataset("explodinggradients/fiqa", "ragas_eval_v3")
eval_data = eval_data["baseline"]
with open('/workspace/data/fiqa/eval_data.jsonl', 'w') as f:
    for l in eval_data:
        f.write(json.dumps(l) + '\n')

corpus = load_dataset("vibrantlabsai/fiqa", "corpus")
corpus = corpus["corpus"]
with open('/workspace/data/fiqa/corpus.jsonl', 'w') as f:
    for l in corpus:
        f.write(json.dumps(l) + '\n')

main = load_dataset("vibrantlabsai/fiqa", "main")
test_data = main["test"]
with open('/workspace/data/fiqa/test.jsonl', 'w') as f:
    for l in test_data:
        f.write(json.dumps(l) + '\n')
train_data = main["train"]
with open('/workspace/data/fiqa/train.jsonl', 'w') as f:
    for l in train_data:
        f.write(json.dumps(l) + '\n')
validation_data = main["validation"]
with open('/workspace/data/fiqa/validation.jsonl', 'w') as f:
    for l in validation_data:
        f.write(json.dumps(l) + '\n')

In [ ]:
def loadfileDB(collection_name, filepath, colname='doc', chunk_size=1000):
    import json
    import pandas as pd
    from qdrant_client import QdrantClient
    from qdrant_client.models import Distance, VectorParams, PointStruct
    from sentence_transformers import SentenceTransformer

    print("\nSetting up Qdrant...")
    client = QdrantClient(host="localhost", port=6333, timeout=10)
    if not client.collection_exists(collection_name):
        print("creating collection...")
        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=384, distance=Distance.COSINE)
        )
    
    embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    global_id = 0
    
    for chunk_idx, df_chunk in enumerate(pd.read_json(filepath, lines=True, chunksize=chunk_size)):
        if ((chunk_idx + 1) % 10 == 0):
            print(f"Processing chunk {chunk_idx + 1}...")
        texts = []
        contexts_list = []
        for idx, row in df_chunk.iterrows():
            contexts = row[colname]
            contexts_list.append(contexts)
            if isinstance(contexts, list):
                context_text = " ".join(str(ctx) for ctx in contexts)
            else:
                context_text = str(contexts)
            texts.append(context_text)
        if texts:
            chunk_embeddings = embedding_model.encode(texts, show_progress_bar=False)
            chunk_points = []
            for j in range(len(chunk_embeddings)):
                chunk_points.append(PointStruct(
                    id=global_id,
                    vector=chunk_embeddings[j].tolist(),
                    payload={colname: contexts_list[j] if isinstance(contexts_list[j], list) else [contexts_list[j]]}
                ))
                global_id += 1
            client.upsert(collection_name=collection_name, points=chunk_points)    
    print(f"\n✅ Completed! Total uploaded {global_id} documents")
    return client

In [ ]:
collection_name="corpus"
client=loadfileDB(collection_name=collection_name, filepath='/workspace/data/fiqa/corpus.jsonl', colname='doc', chunk_size=1000)

In [ ]:
# Get collection info
collection_info = client.get_collection(collection_name)
print(f"Points: {collection_info.points_count}")
print(f"Vectors: {collection_info.config.params.vectors.size}")

In [ ]:
def retrieveQueryEmbeddings(query_texts,client,collection_name, colname='doc', k=3,verbose=True):
    from sentence_transformers import SentenceTransformer
    embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    query_embedding = embedding_model.encode(query_texts).tolist()
    results = client.query_points(collection_name=collection_name,query=query_embedding,limit=k,with_payload=True)
    if verbose:
        for i, point in enumerate(results.points):
            print(f"\n{i+1}. Score: {point.score:.3f}")
            print(f"   ID: {point.id}")
            print(f"   {colname}: {point.payload.get(colname, 'No text')[:150]}...")
    return results

In [ ]:
test_query="Filing personal with 1099s versus business s-corp?"
results=retrieveQueryEmbeddings(query_texts=test_query,
                                client=client,collection_name=collection_name,colname='doc',k=3,verbose=True)

In [ ]:
def llmRespose(query_results, test_query, colname='doc', verbose=True):
    context = "\n".join([f"- {r.payload[colname]}" for r in query_results.points])
    prompt = f"""Based on the following context, answer the question.

    Context:
    {context}

    Question: {test_query}

    Answer:"""

    resp = httpx.post(
        "http://localhost:8001/generate",
        json={"prompt": prompt, "max_tokens": 150, "temperature": 0.7},
        timeout=120.0
    )
    answer=""
    if verbose:
        if resp.status_code == 200:
            answer = resp.json()['text']
            print(f"\nGenerated answer: {answer}")
        else:
            print(f"LLM error: {resp.status_code}")

    return answer

In [ ]:
llmRespose(query_results=results, test_query=test_query, colname='doc', verbose=True)

In [ ]:
# Test LLM generation
print("\nTesting LLM with RAG...")
context = "\n".join([f"- {r.payload['retrieved_contexts']}" for r in results.points])
prompt = f"""Based on the following context, answer the question.

Context:
{context}

Question: {test_query}

Answer:"""

resp = httpx.post(
    "http://localhost:8001/generate",
    json={"prompt": prompt, "max_tokens": 150, "temperature": 0.7},
    timeout=120.0
)

if resp.status_code == 200:
    answer = resp.json()['text']
    print(f"\nGenerated answer: {answer}")
else:
    print(f"LLM error: {resp.status_code}")

print("\n✅ Setup complete! Ready for evaluation.")

In [ ]:
generated_answer = answer[len(prompt):]
ground_truth = df['response'][1]
groundContext=df['retrieved_contexts'][1]

In [ ]:
import evaluate
from sklearn.metrics.pairwise import cosine_similarity


rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")
bleu = evaluate.load("bleu")

# ROUGE scores
rouge_scores = rouge.compute(
    predictions=[generated_answer],
    references=[ground_truth])

# BERTScore
bert_scores = bertscore.compute(
    predictions=[generated_answer],
    references=[ground_truth],
    lang="en")
    
# BLEU score
bleu_scores = bleu.compute(
    predictions=[generated_answer],
    references=[ground_truth])

# Custom: Answer similarity using embeddings
similarity = cosine_similarity([embedding_model.encode(generated_answer)], [embedding_model.encode(ground_truth)])[0][0]

# Custom: Retrieval quality
retrieval_scores = [r.score for r in results.points]
avg_retrieval_score = np.mean(retrieval_scores) if retrieval_scores else 0.0

retrieval_similarity = [cosine_similarity([embedding_model.encode(rc)], embedding_model.encode(groundContext).reshape(1, -1))[0][0] for r in results.points for rc in r.payload['retrieved_contexts']]
avg_retrieval_sim = np.mean(retrieval_similarity) if retrieval_similarity else 0.0

In [ ]:
# Store metrics
metrics = {
    "rouge1": rouge_scores['rouge1'],
    "rouge2": rouge_scores['rouge2'],
    "rougeL": rouge_scores['rougeL'],
    "rougeLsum": rouge_scores['rougeLsum'],
    "bertscore_precision": bert_scores['precision'][0],
    "bertscore_recall": bert_scores['recall'][0],
    "bertscore_f1": bert_scores['f1'][0],
    "bleu": bleu_scores['bleu'],
    "cosine_similarity": float(similarity),
    "avg_retrieval_score": avg_retrieval_score,
    "avg_retrieval_similarity": avg_retrieval_sim
}